# 演習2：小さなGPTを育てて、壊して、覗く（Kaggle 用）

- 右側の「Session options」で **Accelerator を GPU T4 x2**、**Internet を on** にしてください
- 学習は **Save Version（Save & Run All）** で実行してください。対話セッションは途中で止まることがあります。結果は保存したバージョンの「Output」から取り出せます
- 実験を変えるときは、このノートブックではなくリポジトリのコード（`model.py` など）をプルリクエストで変更し、下の `BRANCH` でそのブランチを指定します

## 0. リポジトリを取ってくる

グループのリポジトリは Private なので、GitHub のトークンを使って clone します（作り方は README の「Kaggle から Private リポジトリを読むためのトークン」）。
最初に1回だけ、このノートブックの「Add-ons」→「Secrets」→「Add Secret」で、Label を `GITHUB_TOKEN`、Value をトークンにして登録し、このノートブックで使えるようにチェックを入れてください。

In [ ]:
REPO_URL = "https://github.com/OWNER/REPO.git"  # ← グループのリポジトリに変える
BRANCH = "main"                                # ← 実験したいブランチ
USE_TOKEN = True   # Private リポジトリなので True。公開の配布リポジトリをそのまま試すときだけ False

import os, subprocess
url = REPO_URL
if USE_TOKEN:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN").strip()
    url = REPO_URL.replace("https://", f"https://git:{token}@")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "repo"])
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "repo"], capture_output=True, text=True)
if r.returncode != 0:
    msg = r.stderr.replace(token, "***") if USE_TOKEN else r.stderr  # トークンを画面に出さない
    raise RuntimeError("clone に失敗しました。REPO_URL・BRANCH・トークンを確認してください\n" + msg)
os.chdir("/kaggle/working/repo")
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True))

## 1. データを準備する（30秒ほど）

In [ ]:
!python prepare.py

## 2. 課題①：育てる

標準構成 S を5,000 step（約5〜6分）学習させます。T4 が2枚あるので、seed だけを変えた2本を同時に学習させます。
結果は `runs/<実験名>/` に保存されます。

In [ ]:
%%bash
CUDA_VISIBLE_DEVICES=0 python train.py --name base_seed0 --seed 0 > base_seed0.out 2>&1 &
CUDA_VISIBLE_DEVICES=1 python train.py --name base_seed1 --seed 1 > base_seed1.out 2>&1 &
wait
tail -n 3 base_seed0.out base_seed1.out

### 結果を見る

In [ ]:
import json, glob
import pandas as pd
import matplotlib.pyplot as plt

runs = sorted(glob.glob("runs/*/metrics.json"))
display(pd.DataFrame([json.load(open(p)) for p in runs]))

fig, ax = plt.subplots(figsize=(8, 4))
for p in runs:
    d = p.rsplit("/", 1)[0]
    log = pd.read_csv(f"{d}/log.csv")
    ax.plot(log["step"], log["train_loss"], label=f"{d.split('/')[-1]} train", alpha=0.6)
    v = log.dropna(subset=["val_loss"])
    ax.plot(v["step"], v["val_loss"], "o-", label=f"{d.split('/')[-1]} val")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_ylim(top=6); ax.legend()
plt.show()

In [ ]:
print(open("runs/base_seed0/samples.txt").read())

### 生成してみる（プロンプトや temperature を変えてみる）

In [ ]:
!python generate.py runs/base_seed0 --prompt "吾輩は" --temperature 0.8 --n 3

## 3. 課題②：causal mask を外す

1. 実験の前に、予想を `expectations/causal_mask.md` に書いてプルリクエストで提出する
2. `model.py` を変更したブランチを作り、上の `BRANCH` をそのブランチ名にしてから、このノートブックを最初から実行し直す
3. 下のセルで学習させる（実験名は変更内容がわかるように付ける）

In [ ]:
%%bash
CUDA_VISIBLE_DEVICES=0 python train.py --name nomask_seed0 --seed 0 > nomask_seed0.out 2>&1 &
CUDA_VISIBLE_DEVICES=1 python train.py --name nomask_seed1 --seed 1 > nomask_seed1.out 2>&1 &
wait
tail -n 3 nomask_seed0.out nomask_seed1.out

## 4. （余力があれば）覗く：attention weight を見る

学習したモデルに文を入れ、各層・各ヘッドの attention weight（どの文字にどれだけ注目しているか）をヒートマップにします。
縦が注目する位置、横が見る位置です（講義の causal mask の表と同じ向き）。
`--pos` で指定した位置の文字が、どの文字にどれだけ重みを置いているかも表示します。
causal mask を外したモデルや、担当する比較で変えたモデルでも同じように見て、基準と比べてみてください。

In [ ]:
!python attention.py runs/base_seed0 --text "吾輩は猫である。名前はまだ無い。" --pos 7

In [ ]:
from IPython.display import Image
Image("runs/base_seed0/attention.png")

## 5. 結果をリポジトリに残す

`runs/<実験名>/` の `config.json`、`log.csv`、`samples.txt`、`metrics.json` を、リポジトリの `results/<実験名>/` にコピーしてプルリクエストで残してください。
Save Version で実行した場合は、そのバージョンの「Output」タブの `repo/runs/` からダウンロードできます。

In [ ]:
# 結果だけをまとめた zip を作る（Output からダウンロードしやすくするため。model.pt は含めない）
!cd runs && zip -qr /kaggle/working/results.zip . -x "*/model.pt" && ls -lh /kaggle/working/results.zip